# Portfolio Optimization & Machine Learning Return Forecasting
This notebook focuses purely on the Data Science and Quantitative Finance aspects of the project. We will fetch historical data, run a Markowitz mean-variance optimization, and finally build a Random Forest model to forecast future returns based on momentum and volatility features.


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Aesthetics
sns.set_theme(style="darkgrid")


## 1. Data Fetching
We define our stock universe across various sectors and download historical adjusted close prices.


In [ ]:
STOCK_SECTORS = {
    "Technology": ["AAPL", "MSFT", "NVDA"],
    "Finance": ["JPM", "BAC", "GS"],
    "Energy": ["XOM", "CVX", "COP"],
    "Consumer Staples": ["PG", "KO", "WMT"],
    "Healthcare": ["JNJ", "UNH", "PFE"],
    "Industrial": ["CAT", "BA", "MMM"],
}
ALL_TICKERS = [t for tickers in STOCK_SECTORS.values() for t in tickers]

print(f"Fetching data for {len(ALL_TICKERS)} stocks...")
data = yf.download(ALL_TICKERS, start="2019-04-01", end="2025-03-31", auto_adjust=False, progress=False)
prices = data["Adj Close"]

if hasattr(prices.columns, "get_level_values"):
    prices.columns = prices.columns.get_level_values(0)

monthly_prices = prices.resample("ME").last()
monthly_returns = monthly_prices.pct_change().dropna()
monthly_returns.head()


## 2. Portfolio Optimization (Markowitz Mean-Variance)
We will optimize the portfolio to maximize returns while penalizing variance (risk). We also add constraints so no single stock exceeds 15% of the portfolio.


In [ ]:
# Parameters
risk_aversion = 3  # lambda
max_weight = 0.15

mean_returns = monthly_returns.mean().values
cov_matrix = monthly_returns.cov().values
cov_matrix = (cov_matrix + cov_matrix.T) / 2  # Symmetrize

n_assets = len(ALL_TICKERS)
w = cp.Variable(n_assets)

# Constraints: fully invested, long-only, max position size
constraints = [cp.sum(w) == 1, w >= 0, w <= max_weight]

# Objective
objective = cp.Maximize(mean_returns @ w - 0.5 * risk_aversion * cp.quad_form(w, cp.psd_wrap(cov_matrix)))

# Solve
problem = cp.Problem(objective, constraints)
problem.solve()

optimal_weights = np.maximum(w.value, 0)
optimal_weights /= optimal_weights.sum()

weights_df = pd.DataFrame({"Ticker": ALL_TICKERS, "Weight": optimal_weights})
weights_df = weights_df[weights_df["Weight"] > 0.01].sort_values("Weight", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=weights_df, x="Ticker", y="Weight", palette="viridis")
plt.title("Optimal Portfolio Weights")
plt.ylabel("Weight")
plt.show()


## 3. Machine Learning: Return Forecasting
Here, we build a Random Forest Regressor to predict next month's returns. We engineer features based on historical momentum and volatility.


In [ ]:
# 3.1 Feature Engineering
market_returns = monthly_returns.mean(axis=1)
ml_data = []

for ticker in ALL_TICKERS:
    ret = monthly_returns[ticker]
    features = pd.DataFrame(index=ret.index)
    features["Ticker"] = ticker
    features["Lag_1M"] = ret.shift(1)
    features["Momentum_3M"] = ret.rolling(3).mean().shift(1)
    features["Momentum_6M"] = ret.rolling(6).mean().shift(1)
    features["Volatility_3M"] = ret.rolling(3).std().shift(1)
    features["Volatility_6M"] = ret.rolling(6).std().shift(1)
    features["Market_Return_1M"] = market_returns.shift(1)
    features["Target_Return_1M"] = ret
    ml_data.append(features)

dataset = pd.concat(ml_data).dropna()
print(f"Dataset shape: {dataset.shape}")
dataset.head()


In [ ]:
# 3.2 Walk-Forward Train/Test Split
# We train on data up to 2022, and test on 2023-present to avoid data leakage
train_mask = dataset.index <= "2022-12-31"
test_mask = dataset.index > "2022-12-31"

feature_cols = ["Lag_1M", "Momentum_3M", "Momentum_6M", "Volatility_3M", "Volatility_6M", "Market_Return_1M"]

X_train = dataset.loc[train_mask, feature_cols]
y_train = dataset.loc[train_mask, "Target_Return_1M"]
X_test = dataset.loc[test_mask, feature_cols]
y_test = dataset.loc[test_mask, "Target_Return_1M"]

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")


In [ ]:
# 3.3 Train Random Forest
model = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=8, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
directional_accuracy = (np.sign(y_pred) == np.sign(y_test)).mean()

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Directional Accuracy: {directional_accuracy:.2%}")


In [ ]:
# 3.4 Feature Importance
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importance.values, y=importance.index, palette="mako")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.show()
